# AIS Type 5 메시지 무결성 분석 (테스트용)

두 가지 분석을 수행합니다:
1. **보고 주기 이상 탐지** — 연속 수신 간격이 300~370초 범위를 벗어나는지 확인
2. **선박 재원 유효성 검증** — `vessel_info` 참조 테이블과 비교 (IMO, ship_type, call_sign, vessel_name)

In [1]:
from __future__ import annotations

from datetime import datetime, timezone, timedelta
from difflib import SequenceMatcher
from typing import Optional
import pandas as pd
from cassandra.cluster import Cluster, NoHostAvailable
from cassandra.policies import DCAwareRoundRobinPolicy
from cassandra.query import SimpleStatement

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

## ⚙️ 파라미터 설정
시간 범위와 Cassandra 접속 정보를 여기서 수정하세요.

In [7]:
KST = timezone(timedelta(hours=9))

# 조회 시간 범위 (KST 기준으로 입력)
START_DT = datetime(2026, 4, 21, 0, 0, 0, tzinfo=KST)
END_DT   = datetime(2026, 4, 22, 1, 0, 0, tzinfo=KST)

# Cassandra 접속 정보
CASSANDRA_HOST     = "localhost"
CASSANDRA_PORT     = 9042
CASSANDRA_KEYSPACE = "dlim"
TABLE_STATIC       = "ais_static_voyage"
TABLE_VESSEL_INFO  = "vessel_info"

# 보고 주기 상수
EXPECTED_INTERVAL_SEC    = 360
MAX_INTERVAL_EXPAND0_SEC = 364
MAX_INTERVAL_SEC         = 370
MIN_INTERVAL_SEC         = 300
NAME_SIMILARITY_THRESHOLD = 0.85

print(f"조회 범위: {START_DT.strftime('%Y-%m-%d %H:%M:%S %Z')} ~ {END_DT.strftime('%Y-%m-%d %H:%M:%S %Z')}")

조회 범위: 2026-04-21 00:00:00 UTC+09:00 ~ 2026-04-22 01:00:00 UTC+09:00


## 📡 Cassandra 연결 및 데이터 로드

In [8]:
cluster = Cluster(
    contact_points=[CASSANDRA_HOST],
    port=CASSANDRA_PORT,
    load_balancing_policy=DCAwareRoundRobinPolicy(),
    protocol_version=4,
)
session = cluster.connect(CASSANDRA_KEYSPACE)
print(f"Cassandra 연결 성공: {CASSANDRA_HOST}:{CASSANDRA_PORT} / keyspace={CASSANDRA_KEYSPACE}")

C:\Users\dahun\AppData\Local\Temp\ipykernel_36356\1402166345.py:1: DeprecationWarning: Legacy execution parameters will be removed in 4.0. Consider using execution profiles.
  cluster = Cluster(


Cassandra 연결 성공: localhost:9042 / keyspace=dlim


In [9]:
# ais_static_voyage 로드
cql_static = (
    f"SELECT mmsi, received_at, call_sign, vessel_name, ship_type, imo_number "
    f"FROM {TABLE_STATIC} "
    f"WHERE received_at >= ? AND received_at <= ? "
    f"LIMIT 50000 ALLOW FILTERING"
)
stmt = session.prepare(cql_static)
stmt.fetch_size = 1000

rows = [
    {
        "mmsi":        row.mmsi,
        "received_at": row.received_at,
        "call_sign":   row.call_sign,
        "vessel_name": row.vessel_name,
        "ship_type":   row.ship_type,
        "imo_number":  row.imo_number,
    }
    for row in session.execute(stmt, [
        START_DT.astimezone(timezone.utc).replace(tzinfo=None),
        END_DT.astimezone(timezone.utc).replace(tzinfo=None),
    ])
]
cols = ["mmsi", "received_at", "call_sign", "vessel_name", "ship_type", "imo_number"]
df_static = pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)

print(f"ais_static_voyage 로드: {len(df_static):,}건 / {df_static['mmsi'].nunique() if not df_static.empty else 0}척")
display(df_static.head(10))

ais_static_voyage 로드: 3,616건 / 32척


,mmsi,received_at,call_sign,vessel_name,ship_type,imo_number
0,477348300,2026-04-21 02:00:20.708,VRPN8,SEASPAN BEYOND,71,9739678
1,477348300,2026-04-21 02:00:15.257,VRPN8,SEASPAN BEYOND,71,9739678
2,477348300,2026-04-21 02:00:09.758,VRPN8,SEASPAN BEYOND,71,9739678
3,477348300,2026-04-21 01:00:55.281,VRPN8,SEASPAN BEYOND,71,9739678
4,477348300,2026-04-21 01:00:49.721,VRPN8,SEASPAN BEYOND,71,9739678
5,477348300,2026-04-21 01:00:44.229,VRPN8,SEASPAN BEYOND,71,9739678
6,477348300,2026-04-21 01:00:38.664,VRPN8,SEASPAN BEYOND,71,9739678
7,477348300,2026-04-21 01:00:33.021,VRPN8,SEASPAN BEYOND,71,9739678
8,477348300,2026-04-21 01:00:27.211,VRPN8,SEASPAN BEYOND,71,9739678
9,477348300,2026-04-21 01:00:21.662,VRPN8,SEASPAN BEYOND,71,9739678


In [ ]:
# vessel_info 로드
cql_vi = (
    f"SELECT mmsi, call_sign, vessel_name, ship_type, imo_number, "
    f"dimension_a, dimension_b, dimension_c, dimension_d "
    f"FROM {TABLE_VESSEL_INFO}"
)
stmt_vi = SimpleStatement(cql_vi, fetch_size=5000)
vi_rows = [
    {
        "mmsi":            row.mmsi,
        "ref_call_sign":   row.call_sign,
        "ref_vessel_name": row.vessel_name,
        "ref_ship_type":   row.ship_type,
        "ref_imo_number":  row.imo_number,
        "ref_dim_a":       row.dimension_a,
        "ref_dim_b":       row.dimension_b,
        "ref_dim_c":       row.dimension_c,
        "ref_dim_d":       row.dimension_d,
    }
    for row in session.execute(stmt_vi)
]
vi_cols = ["mmsi", "ref_call_sign", "ref_vessel_name", "ref_ship_type", "ref_imo_number",
           "ref_dim_a", "ref_dim_b", "ref_dim_c", "ref_dim_d"]
df_vessel_info = pd.DataFrame(vi_rows) if vi_rows else pd.DataFrame(columns=vi_cols)

cluster.shutdown()
print(f"vessel_info 로드: {len(df_vessel_info):,}척")
display(df_vessel_info.head(5))

## 📊 분석 1 — 보고 주기 이상 탐지

| 상태 | 조건 | 의미 |
|------|------|------|
| `NORMAL` | 300s ≤ interval ≤ 364s | 정상 |
| `EXPANDED` | 364s < interval ≤ 370s | 슬롯 혼잡 (expand 발생) |
| `LATE` | interval > 370s | 이상 (늦음) |
| `TOO_EARLY` | interval < 300s | 이상 (비정상 재전송) |

In [ ]:
def _classify_interval(sec: float) -> str:
    if sec < MIN_INTERVAL_SEC:
        return "TOO_EARLY"
    if sec <= MAX_INTERVAL_EXPAND0_SEC:
        return "NORMAL"
    if sec <= MAX_INTERVAL_SEC:
        return "EXPANDED"
    return "LATE"


if df_static.empty:
    print("데이터 없음 — 분석 불가")
else:
    df = df_static.copy()
    df["received_at"] = pd.to_datetime(df["received_at"], utc=True, errors="coerce")
    df = df.sort_values(["mmsi", "received_at"])
    df["prev_received_at"] = df.groupby("mmsi")["received_at"].shift(1)
    df_interval = df.dropna(subset=["prev_received_at"]).copy()
    df_interval["interval_sec"] = (
        df_interval["received_at"] - df_interval["prev_received_at"]
    ).dt.total_seconds()
    df_interval["interval_status"] = df_interval["interval_sec"].apply(_classify_interval)

    # MMSI별 메시지 수
    msg_count = df_static.groupby("mmsi").size().rename("msg_count")
    print(f"=== MMSI별 수신 메시지 수 (기대: ~9건/시간) ===")
    display(msg_count.to_frame())

    # 전체 interval 상태 분포
    print("\n=== 보고 주기 분류 결과 ===")
    display(df_interval["interval_status"].value_counts(dropna=False).to_frame("건수"))

In [ ]:
# MMSI별 요약 (메시지 수 + interval 통계 + 상태별 카운트)
if not df_static.empty and not df_interval.empty:
    all_statuses = ["NORMAL", "EXPANDED", "LATE", "TOO_EARLY"]

    status_pivot = (
        df_interval.groupby(["mmsi", "interval_status"])
        .size()
        .unstack(fill_value=0)
    )
    for col in all_statuses:
        if col not in status_pivot.columns:
            status_pivot[col] = 0

    summary = df_interval.groupby("mmsi")["interval_sec"].agg(
        intervals="count",
        mean_sec=lambda x: round(x.mean(), 1),
        min_sec=lambda x: round(x.min(), 1),
        max_sec=lambda x: round(x.max(), 1),
    )
    summary = summary.join(status_pivot[all_statuses], how="left").fillna(0)
    summary = summary.join(msg_count, how="left")
    summary.insert(0, "msg_count", summary.pop("msg_count"))
    summary[all_statuses] = summary[all_statuses].astype(int)

    print("=== MMSI별 보고 주기 요약 ===")
    display(summary)

In [ ]:
# MMSI별 수신 시각 상세 (정상/이상 구분)
if not df_static.empty and not df_interval.empty:
    df_first = df_static.copy()
    df_first["received_at"] = pd.to_datetime(df_first["received_at"], utc=True, errors="coerce")

    for mmsi, grp in df_interval.groupby("mmsi"):
        first_msg = df_first[df_first["mmsi"] == mmsi]["received_at"].min()
        rows_display = []
        if pd.notna(first_msg):
            rows_display.append({
                "received_at (KST)": first_msg.astimezone(KST).strftime("%H:%M:%S"),
                "interval_sec": "-",
                "status": "첫 수신",
            })
        for _, row in grp.iterrows():
            flag = "✓" if row["interval_status"] in ("NORMAL", "EXPANDED") else "✗"
            rows_display.append({
                "received_at (KST)": row["received_at"].astimezone(KST).strftime("%H:%M:%S"),
                "interval_sec": f"{row['interval_sec']:.0f}s",
                "status": f"{flag} {row['interval_status']}",
            })
        print(f"\n--- MMSI {mmsi} ({msg_count.get(mmsi, 0)}건) ---")
        display(pd.DataFrame(rows_display))

## 🚢 분석 2 — 선박 재원 유효성 검증

`ais_static_voyage`에서 수신된 값을 `vessel_info` 참조 테이블과 비교합니다.

In [ ]:
def _normalize_callsign(cs: Optional[str]) -> str:
    if not cs:
        return ""
    return cs.upper().replace("@", "").strip()

def _normalize_imo(imo: Optional[str]) -> str:
    if not imo:
        return ""
    digits = "".join(c for c in str(imo) if c.isdigit())
    return str(int(digits)) if digits else ""

def _name_similarity(a: Optional[str], b: Optional[str]) -> float:
    if not a or not b:
        return 0.0
    a_clean = a.upper().strip().replace("@", "")
    b_clean = b.upper().strip().replace("@", "")
    if b_clean.startswith(a_clean) or a_clean.startswith(b_clean):
        return 1.0
    return SequenceMatcher(None, a_clean, b_clean).ratio()

def _classify_vessel_match(row: pd.Series) -> str:
    if not row.get("ref_mmsi_found", False):
        return "NOT_FOUND"
    issues = []
    imo_ais = _normalize_imo(row.get("imo_number"))
    imo_ref = _normalize_imo(row.get("ref_imo_number"))
    if imo_ais and imo_ref and imo_ais != imo_ref:
        issues.append("IMO_MISMATCH")
    stype_ais = row.get("ship_type")
    stype_ref = row.get("ref_ship_type")
    if stype_ais is not None and stype_ref is not None and int(stype_ais) != int(stype_ref):
        issues.append("SHIPTYPE_MISMATCH")
    cs_ais = _normalize_callsign(row.get("call_sign"))
    cs_ref = _normalize_callsign(row.get("ref_call_sign"))
    if cs_ais and cs_ref and cs_ais != cs_ref:
        issues.append("CALLSIGN_MISMATCH")
    sim = _name_similarity(row.get("vessel_name"), row.get("ref_vessel_name"))
    if sim < NAME_SIMILARITY_THRESHOLD:
        issues.append(f"NAME_MISMATCH(sim={sim:.2f})")
    elif sim < 1.0:
        issues.append(f"NAME_SIMILAR(sim={sim:.2f})")
    return "|".join(issues) if issues else "OK"


if df_static.empty or df_vessel_info.empty:
    print("데이터 없음 — 분석 불가")
else:
    df_latest = (
        df_static.sort_values("received_at")
        .groupby("mmsi").last().reset_index()
    )
    df_merged = df_latest.merge(df_vessel_info, on="mmsi", how="left")
    df_merged["ref_mmsi_found"] = (
        df_merged["ref_call_sign"].notna() | df_merged["ref_vessel_name"].notna()
    )
    df_merged["match_result"] = df_merged.apply(_classify_vessel_match, axis=1)

    print("=== 재원 검증 결과 분류 ===")
    display(df_merged["match_result"].value_counts().to_frame("건수"))

In [ ]:
# 불일치 상세
if not df_static.empty and not df_vessel_info.empty:
    df_issues = df_merged[df_merged["match_result"] != "OK"]
    if df_issues.empty:
        print("모든 선박 재원 검증 통과 ✓")
    else:
        DETAIL_COLS = [
            "mmsi", "vessel_name", "ref_vessel_name",
            "call_sign", "ref_call_sign",
            "imo_number", "ref_imo_number",
            "ship_type", "ref_ship_type",
            "match_result",
        ]
        print(f"=== 재원 불일치 {len(df_issues)}건 ===")
        display(df_issues[DETAIL_COLS].reset_index(drop=True))